# BME 507 — Warm-up Exercise 5: Semantic Lung Segmentation

**Goal:** segment both lungs as one semantic class (lung = 1, background = 0).

**Simple experiment:** train the same small U-Net twice:
1. Binary cross-entropy (BCE)
2. BCE + Dice loss

The notebook automatically saves:
- training histories (`.csv`)
- final test metrics (`.csv`)
- per-image test metrics (`.csv`)
- model weights (`.pth`)
- training curves (`.png`)
- qualitative segmentation examples (`.png`)

Everything is saved under:

`MyDrive/BME507/Warmup_5_6_7/`

In [ ]:
# ============================================================
# BME 507 - WARM-UP EXERCISE 5
# Semantic lung segmentation
# Experiment: BCE vs BCE + Dice loss
# ============================================================

import os
import random
import shutil
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

# ------------------------------------------------------------
# 1. SETTINGS + GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/BME507/Warmup_5_6_7")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = Path("/content/Segmentation01.zip")
EXTRACT_DIR = Path("/content/task5_segmentation01")

DATA_URLS = [
    "https://imgcom.jsrt.or.jp/imgcom/wp-content/uploads/2018/11/Segmentation01.zip",
    "http://imgcom.jsrt.or.jp/imgcom/wp-content/uploads/2018/11/Segmentation01.zip",
]

IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 20
LEARNING_RATE = 1e-3
PATIENCE = 5
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. GET + EXTRACT DATASET
# ------------------------------------------------------------

def download_if_needed(zip_path, urls):
    if zip_path.exists() and zip_path.stat().st_size > 1000:
        print("Dataset zip already exists:", zip_path)
        return

    for url in urls:
        print("Trying:", url)
        result = subprocess.run(
            ["wget", "-q", "-O", str(zip_path), url],
            capture_output=True
        )
        if result.returncode == 0 and zip_path.exists() and zip_path.stat().st_size > 1000:
            print("Downloaded:", zip_path)
            return

    raise FileNotFoundError(
        "Could not download Segmentation01.zip automatically. "
        "Upload Segmentation01.zip manually to /content/Segmentation01.zip and rerun."
    )

def find_named_dir(root, name):
    for current, dirs, files in os.walk(root):
        if Path(current).name.lower() == name.lower():
            return Path(current)
    raise FileNotFoundError(f"Folder '{name}' was not found under {root}")

def pair_images_and_masks(image_dir, mask_dir):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

    images = {
        p.stem: p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    }
    masks = {
        p.stem: p for p in mask_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    }

    common = sorted(set(images) & set(masks))
    if not common:
        raise RuntimeError(
            f"No matching image/mask filenames found in:\n{image_dir}\n{mask_dir}"
        )

    return [(images[k], masks[k]) for k in common]

download_if_needed(ZIP_PATH, DATA_URLS)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

org_train = find_named_dir(EXTRACT_DIR, "org_train")
label_train = find_named_dir(EXTRACT_DIR, "label_train")
org_test = find_named_dir(EXTRACT_DIR, "org_test")
label_test = find_named_dir(EXTRACT_DIR, "label_test")

all_train_pairs = pair_images_and_masks(org_train, label_train)
test_pairs = pair_images_and_masks(org_test, label_test)

print("Training pairs:", len(all_train_pairs))
print("Official test pairs:", len(test_pairs))

# Reproducible train/validation split from the official training set
indices = np.arange(len(all_train_pairs))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n_val = max(1, int(round(0.20 * len(indices))))
val_indices = indices[:n_val]
train_indices = indices[n_val:]

train_pairs = [all_train_pairs[i] for i in train_indices]
val_pairs = [all_train_pairs[i] for i in val_indices]

print("Train:", len(train_pairs), "| Validation:", len(val_pairs), "| Test:", len(test_pairs))

# ------------------------------------------------------------
# 3. DATASET
# ------------------------------------------------------------

class BinaryLungDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR
        )
        mask = Image.open(mask_path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST
        )

        image = np.asarray(image, dtype=np.float32) / 255.0
        mask = np.asarray(mask, dtype=np.uint8)
        mask = (mask >= 128).astype(np.float32)

        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask, image_path.name

train_loader = DataLoader(
    BinaryLungDataset(train_pairs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    BinaryLungDataset(val_pairs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

test_dataset = BinaryLungDataset(test_pairs)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

# ------------------------------------------------------------
# 4. SMALL U-NET
# ------------------------------------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base=16):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = DoubleConv(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)

        self.out = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)

# ------------------------------------------------------------
# 5. LOSSES + METRICS
# ------------------------------------------------------------

bce_loss = nn.BCEWithLogitsLoss()

def soft_dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    denominator = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()

def get_loss(logits, targets, loss_name):
    if loss_name == "BCE":
        return bce_loss(logits, targets)
    if loss_name == "BCE_Dice":
        return bce_loss(logits, targets) + soft_dice_loss(logits, targets)
    raise ValueError(loss_name)

def binary_metrics_from_logits(logits, targets, eps=1e-7):
    pred = (torch.sigmoid(logits) >= 0.5).float()

    intersection = (pred * targets).sum(dim=(1, 2, 3))
    pred_sum = pred.sum(dim=(1, 2, 3))
    target_sum = targets.sum(dim=(1, 2, 3))
    union = pred_sum + target_sum - intersection

    dice = (2 * intersection + eps) / (pred_sum + target_sum + eps)
    iou = (intersection + eps) / (union + eps)
    acc = (pred == targets).float().mean(dim=(1, 2, 3))

    return dice, iou, acc

# ------------------------------------------------------------
# 6. TRAINING
# ------------------------------------------------------------

def evaluate_loader(model, loader, loss_name):
    model.eval()

    losses, dices, ious, accs = [], [], [], []

    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = get_loss(logits, masks, loss_name)

            dice, iou, acc = binary_metrics_from_logits(logits, masks)

            losses.append(loss.item())
            dices.extend(dice.cpu().numpy())
            ious.extend(iou.cpu().numpy())
            accs.extend(acc.cpu().numpy())

    return {
        "loss": float(np.mean(losses)),
        "dice": float(np.mean(dices)),
        "iou": float(np.mean(ious)),
        "pixel_accuracy": float(np.mean(accs)),
    }

def train_experiment(experiment_name, loss_name):
    model = UNet(in_channels=1, out_channels=1, base=16).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = []
    best_val_dice = -1.0
    best_state = None
    patience_counter = 0

    print("\n" + "=" * 70)
    print("TRAINING:", experiment_name)
    print("=" * 70)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for images, masks, _ in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = get_loss(logits, masks, loss_name)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            seen += images.size(0)

        train_loss = running_loss / seen
        val = evaluate_loader(model, val_loader, loss_name)

        history.append({
            "experiment": experiment_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val["loss"],
            "val_dice": val["dice"],
            "val_iou": val["iou"],
            "val_pixel_accuracy": val["pixel_accuracy"],
        })

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_dice={val['dice']:.4f} | "
            f"val_iou={val['iou']:.4f}"
        )

        if val["dice"] > best_val_dice:
            best_val_dice = val["dice"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    weight_path = SAVE_DIR / f"task5_{experiment_name}_best.pth"
    torch.save(model.state_dict(), weight_path)

    print("Saved model:", weight_path)
    return model, pd.DataFrame(history)

# ------------------------------------------------------------
# 7. RUN BOTH EXPERIMENTS
# ------------------------------------------------------------

experiments = [
    ("BCE", "BCE"),
    ("BCE_Dice", "BCE_Dice"),
]

models = {}
history_frames = []
summary_rows = []
per_image_frames = []

for experiment_name, loss_name in experiments:
    model, history_df = train_experiment(experiment_name, loss_name)

    models[experiment_name] = model
    history_frames.append(history_df)

    test_metrics = evaluate_loader(model, test_loader, loss_name)

    summary_rows.append({
        "experiment": experiment_name,
        "loss_function": loss_name,
        "test_dice": test_metrics["dice"],
        "test_iou": test_metrics["iou"],
        "test_pixel_accuracy": test_metrics["pixel_accuracy"],
        "epochs_completed": int(history_df["epoch"].max()),
        "best_val_dice": float(history_df["val_dice"].max()),
    })

    # Per-image metrics
    rows = []
    model.eval()
    with torch.no_grad():
        for image, mask, filename in test_dataset:
            image_b = image.unsqueeze(0).to(device)
            mask_b = mask.unsqueeze(0).to(device)

            logits = model(image_b)
            dice, iou, acc = binary_metrics_from_logits(logits, mask_b)

            rows.append({
                "experiment": experiment_name,
                "filename": filename,
                "dice": float(dice.item()),
                "iou": float(iou.item()),
                "pixel_accuracy": float(acc.item()),
            })

    per_image_frames.append(pd.DataFrame(rows))

# ------------------------------------------------------------
# 8. SAVE NUMERICAL RESULTS
# ------------------------------------------------------------

history_all = pd.concat(history_frames, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
per_image_df = pd.concat(per_image_frames, ignore_index=True)

history_all.to_csv(SAVE_DIR / "task5_training_history.csv", index=False)
summary_df.to_csv(SAVE_DIR / "task5_summary.csv", index=False)
per_image_df.to_csv(SAVE_DIR / "task5_test_per_image.csv", index=False)

print("\nFINAL TASK 5 RESULTS")
print(summary_df)

# ------------------------------------------------------------
# 9. SAVE TRAINING CURVES
# ------------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name in summary_df["experiment"]:
    d = history_all[history_all["experiment"] == name]
    plt.plot(d["epoch"], d["train_loss"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Task 5 - Training loss")
plt.legend()

plt.subplot(1, 2, 2)
for name in summary_df["experiment"]:
    d = history_all[history_all["experiment"] == name]
    plt.plot(d["epoch"], d["val_dice"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation Dice")
plt.title("Task 5 - Validation Dice")
plt.legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / "task5_training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 10. SAVE QUALITATIVE TEST EXAMPLES
# ------------------------------------------------------------

n_examples = min(4, len(test_dataset))
example_indices = np.linspace(0, len(test_dataset) - 1, n_examples, dtype=int)

plt.figure(figsize=(12, 3 * n_examples))

for row, idx in enumerate(example_indices):
    image, mask, filename = test_dataset[idx]

    with torch.no_grad():
        pred_bce = (
            torch.sigmoid(models["BCE"](image.unsqueeze(0).to(device))) >= 0.5
        ).float().cpu().squeeze().numpy()

        pred_dice = (
            torch.sigmoid(models["BCE_Dice"](image.unsqueeze(0).to(device))) >= 0.5
        ).float().cpu().squeeze().numpy()

    panels = [
        (image.squeeze().numpy(), f"X-ray\n{filename}", "gray"),
        (mask.squeeze().numpy(), "Ground truth", "gray"),
        (pred_bce, "BCE prediction", "gray"),
        (pred_dice, "BCE + Dice prediction", "gray"),
    ]

    for col, (arr, title, cmap) in enumerate(panels):
        ax = plt.subplot(n_examples, 4, row * 4 + col + 1)
        ax.imshow(arr, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.savefig(SAVE_DIR / "task5_examples.png", dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 11. SAVE RUN INFORMATION
# ------------------------------------------------------------

run_info = [
    f"device={device}",
    f"gpu={torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}",
    f"image_size={IMAGE_SIZE}",
    f"batch_size={BATCH_SIZE}",
    f"max_epochs={EPOCHS}",
    f"learning_rate={LEARNING_RATE}",
    f"train_images={len(train_pairs)}",
    f"validation_images={len(val_pairs)}",
    f"test_images={len(test_pairs)}",
]

(SAVE_DIR / "task5_run_info.txt").write_text("\n".join(run_info))

print("\nSaved Task 5 outputs to:", SAVE_DIR)